# Tarvo · a conta de setembro/2026 e as hipóteses

Este notebook lê **somente** as tabelas padronizadas em `data/` (geradas por `python -m src.ingestao`) e faz duas coisas:

1. **Mostra a conta do mês**: o que vendeu, o que a Orbe cobrou, quanto a Tarvo recebeu e pagou nos bancos, e o que se ganha nas duas linhas de negócio.
2. **Testa as hipóteses** do `HIPOTESES.md` e as dúvidas que surgiram na exploração, uma a uma, com o número que sustenta cada conclusão.

Regra do notebook: o que é **fato dos arquivos** aparece como número. O que é **interpretação** vem marcado como tal. Valores de comissão aqui são **prévias** para testar hipóteses; os arquivos oficiais de saída são outra etapa.

In [1]:
from decimal import Decimal, ROUND_DOWN, ROUND_HALF_EVEN, ROUND_HALF_UP
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_rows", 120)

DATA = Path("data")
assert DATA.exists(), "Rode antes: python -m src.ingestao --mes 2026-09 --entrada . --saida data"


def ler(nome, num=(), datas=()):
    df = pd.read_csv(DATA / f"{nome}.csv", sep=";", dtype=str, keep_default_na=False)
    for c in num:
        df[c] = pd.to_numeric(df[c].replace("", pd.NA))
    for c in datas:
        df[c] = pd.to_datetime(df[c].replace("", pd.NA), format="%Y-%m-%d")
    return df


TAXAS = ["taxa_debito_pct", "taxa_credito_vista_pct", "taxa_credito_2_6x_pct", "taxa_credito_7_12x_pct",
         "taxa_pix_pct", "taxa_antecipacao_pct"]
lojista = ler("dim_lojista")
ec = ler("dim_ec", num=["n_linhas_orbe"])
contrato = ler("dim_contrato_taxa", num=TAXAS + ["spread_remessa_pct", "spread_remessa_efetivo_pct"],
               datas=["vigencia_ini", "vigencia_fim"])
plano = ler("dim_plano_contas")
orbe = ler("fato_orbe", num=["parcelas", "valor_bruto", "taxa_aplicada_pct", "valor_taxa", "valor_liquido"],
           datas=["data_movimento", "data_venda_original", "data_taxa_ref"])
ponte = ler("fato_ponte", num=["valor_me", "taxa_cliente", "ptax", "valor_brl", "spread_brl", "spread_pct",
                               "spread_estimado_brl", "iof_brl", "spread_contratado_pct"],
            datas=["data_contratacao", "data_liquidacao"])
banco = ler("fato_banco", num=["valor", "saldo"], datas=["data"])
controle = ler("controle_ingestao")

print(pd.DataFrame({"tabela": ["dim_lojista", "dim_ec", "dim_contrato_taxa", "dim_plano_contas", "fato_orbe",
                               "fato_ponte", "fato_banco"],
                    "linhas": [len(t) for t in (lojista, ec, contrato, plano, orbe, ponte, banco)],
                    "colunas": [t.shape[1] for t in (lojista, ec, contrato, plano, orbe, ponte, banco)]}).to_string(index=False))

           tabela  linhas  colunas
      dim_lojista      25       12
           dim_ec      23       15
dim_contrato_taxa      27       24
 dim_plano_contas      52        8
        fato_orbe    1943       23
       fato_ponte     233       28
       fato_banco     184       16


### Verificações da ingestão
Antes de olhar qualquer número, conferir se a ingestão confiou nos próprios dados.

In [2]:
verif = controle[controle.tipo == "VERIFICACAO"][["fonte", "item", "valor", "detalhe"]]
display(verif)
print("Alertas:", int((verif.valor == "ALERTA").sum()), "de", len(verif), "verificações")

,fonte,item,valor,detalhe
20,aurora,saldo_final_recalculado_igual_ao_informado,OK,calculado=-1920088.6; informado=-1920088.6
21,aurora,saldo_linha_a_linha_confere,OK,linhas_com_diferenca=0
26,meridiano,fitid_unico,OK,duplicados=0
31,dim_ec,dim_ec_cobre_orbe_tabela_e_config,OK,ECs no universo=23; na dim_ec=23
32,dim_ec,ecs_respondidos_pelo_financeiro_com_status_esperado,OK,"1043300981=CONFIRMADO, 1043396662=CONFIRMADO, 1038867654=CONFIRMADO, 1076801732=RETIDO"
34,orbe,id_transacao_unico,OK,duplicados=0
35,orbe,valor_taxa_igual_bruto_x_taxa_truncado,OK,linhas_com_diferenca=0 (a Orbe trunca em centavos)
36,orbe,liquido_igual_bruto_menos_taxa,OK,linhas_com_diferenca=0
46,ponte,linhas_lidas_igual_total_declarado,OK,lidas=233; declarado=233
47,ponte,soma_spread_igual_total_declarado,OK,soma=78073.59; declarado=78073.59


Alertas: 1 de 18 verificações


### O que é "EC"
**EC** é a sigla de **Estabelecimento Comercial**: no mundo de cartões, é o cadastro do vendedor junto à adquirente. Aqui, o `cod_estabelecimento` é o **código com que a Orbe identifica cada lojista da Tarvo**. Um lojista tem um EC; o relatório da Orbe traz um EC por linha, mas escreve o **nome** do lojista de várias maneiras. Por isso o EC é a chave de ligação com o contrato, e o nome não.

A tabela `dim_ec` guarda, para cada EC visto na Orbe, **a quem ele pertence e se isso está confirmado**:

In [3]:
ec[["ec", "status", "faturavel", "cnpj", "nome_tabela", "nomes_orbe", "n_linhas_orbe", "fonte_mapeamento",
    "sugestao_nome_tabela"]]

,ec,status,faturavel,cnpj,nome_tabela,nomes_orbe,n_linhas_orbe,fonte_mapeamento,sugestao_nome_tabela
0,1000960410,CONFIRMADO,S,23109187000161,Oxford Prep Cursos Ltda,OXFORD PREP CURSOS,98,tabela_taxas,
1,1014537966,CONFIRMADO,S,68567965000102,Dublin Hub Intercâmbio Ltda,DUBLIN HUB INTERCAMBIO,111,tabela_taxas,
2,1018325654,CONFIRMADO,S,00520815000100,Casa do Estudante Viagens Ltda,CASA DO ESTUDANTE VIAG | CASA DO ESTUDANTE VIAGENS,119,tabela_taxas,
3,1020660828,CONFIRMADO,S,16181276000153,EduAbroad,EDU ABROAD CONSULTORIA | EDUABROAD,100,tabela_taxas,
4,1023622151,CONFIRMADO,S,61720397000180,Lisboa Cursos e Viagens Ltda,LISBOA CURSOS E VIAGENS | LISBOA CURSOS VIAGENS,76,tabela_taxas,
5,1037647039,CONFIRMADO,S,17217888000111,Mundo Aberto,MUNDO ABERTO INTERC LTDA | MUNDO ABERTO INTERCAMBIO | Mundo Aberto,168,tabela_taxas,
6,1038867654,CONFIRMADO,S,07752116000126,Kiwi Experience,KIWI EXPERIENCE TURISMO,85,config/mapeamento_ec.csv,
7,1041180753,CONFIRMADO,S,34671260000100,Maple Study Assessoria Educacional Ltda,MAPLE STUDY | MAPLE STUDY ASSESSORIA,105,tabela_taxas,
8,1043300981,CONFIRMADO,S,66564512000199,Novo Horizonte Intercambio,HORIZONTE INTERCAMBIO,73,config/mapeamento_ec.csv,
9,1043396662,CONFIRMADO,S,78576234000179,Horizonte Viagens,HORIZONTE VIAGENS TURISMO,75,config/mapeamento_ec.csv,


## 1. A conta de setembro

### 1.1 Checkout: o que a Orbe reportou
Cada linha da Orbe é uma venda, um estorno, uma antecipação ou uma reversão de antecipação.

In [4]:
tipos = (orbe.groupby("tipo_lancamento")
         .agg(linhas=("id_transacao", "size"), bruto=("valor_bruto", "sum"), taxa_orbe=("valor_taxa", "sum"),
              liquido=("valor_liquido", "sum")).round(2))
tipos.loc["TOTAL"] = tipos.sum()
display(tipos)
vendas = orbe[orbe.tipo_lancamento == "VENDA"]
print("\nVendas por faixa de taxa:")
display(vendas.groupby("faixa").agg(linhas=("id_transacao", "size"), bruto=("valor_bruto", "sum"),
                                    taxa_aplicada_media_pct=("taxa_aplicada_pct", "mean")).round(2))

,linhas,bruto,taxa_orbe,liquido
tipo_lancamento,,,,
ANTECIPACAO,38.0,1451981.71,30491.41,1421490.30
ESTORNO,59.0,-216346.03,-7108.69,-209237.34
REVERSAO ANTECIPACAO,6.0,-288606.34,-6060.69,-282545.65
VENDA,1840.0,6549199.22,214558.33,6334640.89
TOTAL,1943.0,7496228.56,231880.36,7264348.20



Vendas por faixa de taxa:


,linhas,bruto,taxa_aplicada_media_pct
faixa,,,
credito_2_6x,597,2579017.05,3.19
credito_7_12x,323,2738539.84,3.80
credito_a_vista,575,966500.34,2.66
debito,168,82523.09,1.49
pix,177,182618.90,0.89


In [5]:
nome_ec = ec.set_index("ec").apply(lambda r: r.nome_tabela or r.nomes_orbe.split(" | ")[0], axis=1)
status_ec = ec.set_index("ec").status
por_ec = (orbe.groupby("ec")
          .agg(linhas=("id_transacao", "size"), bruto_vendas=("valor_bruto", lambda s: s[orbe.loc[s.index, "tipo_lancamento"] == "VENDA"].sum()),
               taxa_orbe=("valor_taxa", "sum")).round(2))
por_ec.insert(0, "lojista", por_ec.index.map(nome_ec))
por_ec.insert(1, "status_ec", por_ec.index.map(status_ec))
por_ec.sort_values("bruto_vendas", ascending=False)

,lojista,status_ec,linhas,bruto_vendas,taxa_orbe
ec,,,,,
1037647039,Mundo Aberto,CONFIRMADO,168,579638.19,18521.38
1074504011,Rota Global Intercâmbio Ltda,CONFIRMADO,159,511161.89,17166.83
1018325654,Casa do Estudante Viagens Ltda,CONFIRMADO,119,386422.48,15391.27
1041180753,Maple Study Assessoria Educacional Ltda,CONFIRMADO,105,380418.84,15010.25
1075624862,Escola Bridge de Idiomas Ltda,CONFIRMADO,107,373479.93,11945.46
1020660828,EduAbroad,CONFIRMADO,100,337240.08,12069.64
1000960410,Oxford Prep Cursos Ltda,CONFIRMADO,98,312600.49,16077.42
1038867654,Kiwi Experience,CONFIRMADO,85,307366.51,10711.06
1014537966,Dublin Hub Intercâmbio Ltda,CONFIRMADO,111,301207.30,11905.52


### 1.2 Remessa: o que a Ponte reportou
Contas feitas sem a operação repetida (`flag_duplicata = S`), que entra uma vez só. Spread negativo fica fora da comissão (regra do financeiro); a comissão é **50% do spread realizado**, arredondada por operação.

In [6]:
op = ponte[ponte.flag_duplicata != "S"].copy()
op["comissao_tarvo"] = [float((Decimal(str(s)) * Decimal("0.5")).quantize(Decimal("0.01"), ROUND_HALF_UP)) if s > 0 else 0.0
                        for s in op.spread_brl]
por_moeda = (op.groupby("moeda")
             .agg(operacoes=("num_op", "size"), valor_me=("valor_me", "sum"), valor_brl=("valor_brl", "sum"),
                  spread_brl=("spread_brl", "sum"), comissao_tarvo=("comissao_tarvo", "sum"), iof=("iof_brl", "sum")).round(2))
por_moeda.loc["TOTAL"] = por_moeda.sum()
display(por_moeda)
print("Operações únicas:", len(op), "| spreads negativos (fora da comissão):", int((op.spread_brl < 0).sum()),
      "| 'com ressalva' (incluídas):", int((op.flag_ressalva == "S").sum()))

,operacoes,valor_me,valor_brl,spread_brl,comissao_tarvo,iof
moeda,,,,,,
AUD,48.0,234433.14,839951.54,11329.45,5837.37,3191.81
CAD,42.0,202407.79,808431.03,10705.21,5607.31,3072.06
EUR,33.0,187134.41,1197532.31,16074.48,8381.08,4550.62
GBP,36.0,169456.27,1232906.53,17739.02,8869.58,4685.05
NZD,37.0,185680.20,604128.34,8218.05,4255.26,2295.70
USD,36.0,165915.37,907254.58,13185.50,6592.85,3447.55
TOTAL,232.0,1145027.18,5590204.33,77251.71,39543.45,21242.79


Operações únicas: 232 | spreads negativos (fora da comissão): 4 | 'com ressalva' (incluídas): 16


### 1.3 Bancos: o que a Tarvo recebeu e pagou
A categoria abaixo é **exploratória** (ajuda a ler o extrato). Não é a classificação no plano de contas.

In [7]:
def categoria(r):
    t, c, d = r.tipo_lancamento, r.contraparte.upper(), r.descricao.upper()
    if r.banco == "AURORA":
        if "ORBE" in c: return "Orbe: TED recebido"
        if t == "PIX ENVIADO" and "CONTABILIDADE" in c: return "Fornecedor: contabilidade"
        if t == "PIX ENVIADO": return "Repasse PIX a lojistas"
        if c.startswith("TARVO"): return "Transferência entre contas próprias"
        if t in ("PAGTO SALARIOS", "GPS INSS", "GRF FGTS"): return "Folha e encargos"
        if t in ("DEBITO DARF", "PAGTO ISS"): return "Impostos"
        if t in ("BOLETO CONDOMINIO", "DEB AUT", "CARTAO PJ"): return "Despesas operacionais"
        if t.startswith("TAR "): return "Tarifas bancárias"
        if t in ("PIX RECEBIDO", "DEVOLUCAO PIX"): return "Crédito sem identificação / devolução"
    else:
        if t == "PIX RECEB": return "PIX recebido de lojistas (remessa)"
        if t == "TED ENVIADA": return "TED para a Ponte (liquidação)"
        if t == "TED RECEB": return "Comissão recebida da Ponte"
        if c.startswith("TARVO"): return "Transferência entre contas próprias"
        if t in ("APLICACAO AUTOMATICA", "RESGATE AUTOMATICO", "RENDIMENTO APLIC"): return "Aplicação, resgate e rendimento"
        if t.startswith("TARIFA") or t.startswith("SEGURO") or t.startswith("IOF"): return "Tarifas, seguro e IOF"
    return "Outros"

banco["categoria"] = banco.apply(categoria, axis=1)
resumo_banco = (banco.pivot_table(index=["banco", "categoria"], columns="sentido", values="valor", aggfunc="sum", fill_value=0)
                .round(2))
resumo_banco["lancamentos"] = banco.groupby(["banco", "categoria"]).size()
display(resumo_banco)
print("Sem categoria ('Outros'):", int((banco.categoria == "Outros").sum()), "lançamentos")
tot = banco.groupby(["banco", "sentido"]).valor.sum().unstack().round(2)
tot["liquido"] = tot.sum(axis=1)
display(tot)

sentido                                             ENTRADA       SAIDA  lancamentos
banco     categoria                                                                 
AURORA    Crédito sem identificação / devolução    19750.00        0.00            2
          Despesas operacionais                        0.00   -71868.77            3
          Folha e encargos                             0.00  -250831.41            3
          Fornecedor: contabilidade                    0.00    -6800.00            1
          Impostos                                     0.00   -99075.41            4
          Orbe: TED recebido                     3557246.29        0.00           21
          Repasse PIX a lojistas                       0.00 -4230267.61           44
          Tarifas bancárias                            0.00     -576.50            2
          Transferência entre contas próprias          0.00 -1450000.00            4
MERIDIANO Aplicação, resgate e rendimento         476793.66  -588124.00            7
          Comissão recebida da Ponte               61884.12        0.00            1
          PIX recebido de lojistas (remessa)     5629687.12        0.00           64
          TED para a Ponte (liquidação)                0.00 -5611447.12           21
          Tarifas, seguro e IOF                        0.00    -2458.55            4
          Transferência entre contas próprias    1150000.00        0.00            3

Sem categoria ('Outros'): 0 lançamentos


sentido,ENTRADA,SAIDA,liquido
banco,,,
AURORA,3576996.29,-6109419.70,-2532423.41
MERIDIANO,7318364.90,-6202029.67,1116335.23


In [8]:
au = banco[banco.banco == "AURORA"].copy()
saldo_dia = au.groupby("data").saldo.last()
negativos = saldo_dia[saldo_dia < 0]
print(f"Aurora: saldo inicial R$ {float(controle.loc[(controle.fonte=='aurora')&(controle.item=='saldo_anterior'),'valor'].iloc[0]):,.2f}"
      f" | final R$ {saldo_dia.iloc[-1]:,.2f} | menor R$ {saldo_dia.min():,.2f} em {saldo_dia.idxmin().date()}")
print(f"Primeiro dia no vermelho: {negativos.index.min().date()} | dias no vermelho: {len(negativos)} de {len(saldo_dia)}")
saldo_dia.round(2).to_frame("saldo_fim_do_dia")

Aurora: saldo inicial R$ 612,334.81 | final R$ -1,920,088.60 | menor R$ -2,289,532.94 em 2026-09-24
Primeiro dia no vermelho: 2026-09-09 | dias no vermelho: 16 de 22


,saldo_fim_do_dia
data,
2026-09-01,677971.81
2026-09-02,537601.84
2026-09-03,708356.54
2026-09-04,624848.47
2026-09-05,596398.47
2026-09-08,133838.34
2026-09-09,-100597.44
2026-09-10,-1041759.94
2026-09-11,-1276574.84


### 1.4 O que a Tarvo "ganhou" em setembro (prévia)
Aplicando as regras já confirmadas pelo financeiro: checkout pela **taxa contratada** (bruto × taxa da planilha, na data da venda; estorno usa a data da venda original), arredondamento comum por operação, só ECs com mapeamento confirmado, Atlas retida.

In [9]:
FAIXA_COL = {"debito": "taxa_debito_pct", "credito_a_vista": "taxa_credito_vista_pct", "credito_2_6x": "taxa_credito_2_6x_pct",
             "credito_7_12x": "taxa_credito_7_12x_pct", "pix": "taxa_pix_pct", "antecipacao": "taxa_antecipacao_pct"}


def comissao(bruto, taxa):
    x = Decimal(str(bruto)) * Decimal(str(taxa)) / Decimal(100)
    sinal = -1 if x < 0 else 1
    return sinal * abs(x).quantize(Decimal("0.01"), ROUND_HALF_UP)


conf = orbe[orbe.status_ec == "CONFIRMADO"]
base = conf.merge(contrato[["ec", "vigencia_ini", "vigencia_fim"] + TAXAS], on="ec", how="left")
vig_ok = ((base.vigencia_ini.isna() | (base.vigencia_ini <= base.data_taxa_ref))
          & (base.vigencia_fim.isna() | (base.data_taxa_ref <= base.vigencia_fim)))
base = base[vig_ok].copy()
assert len(base) == len(conf), f"vigência deixou {len(conf) - len(base)} linhas sem contrato ou duplicou"
base["taxa_contrato_pct"] = base.apply(lambda r: r[FAIXA_COL[r.faixa]], axis=1)
base["comissao_contrato"] = [float(comissao(b, t)) for b, t in zip(base.valor_bruto, base.taxa_contrato_pct)]
# Regra do financeiro: divergência existe quando a taxa aplicada é diferente da contratada, e o valor é
# valor bruto × (taxa aplicada − contratada), arredondado por linha. Centavo de truncamento da Orbe NÃO é divergência.
base["delta_pp"] = (base.taxa_aplicada_pct - base.taxa_contrato_pct).round(4)
base["divergente"] = base.delta_pp.abs() > 1e-6
base["dif_brl"] = [float(comissao(b, d)) if dv else 0.0 for b, d, dv in zip(base.valor_bruto, base.delta_pp, base.divergente)]
base["dif_trunc_orbe"] = (base.valor_taxa - base.comissao_contrato).round(2)   # só informativo (H1)
base["linha"] = base.tipo_lancamento.map(lambda t: "antecipacao" if "ANTECIPACAO" in t else "MDR")
print("Linhas com contrato encontrado:", len(base), "de", len(conf), "(ECs confirmados)")
print("Linhas fora da comissão por EC não confirmado:", int((orbe.status_ec != "CONFIRMADO").sum()),
      "(", ", ".join(sorted(orbe[orbe.status_ec != "CONFIRMADO"].status_ec.unique())), ")")
base["lojista"] = base.cnpj.map(lojista.set_index("cnpj").nome_canonico)
com_lojista = (base.pivot_table(index="lojista", columns="linha", values="comissao_contrato", aggfunc="sum", fill_value=0).round(2))
com_lojista["total"] = com_lojista.sum(axis=1).round(2)
display(com_lojista.sort_values("total", ascending=False))
print(f"\nCheckout (taxa contratada, ECs confirmados): R$ {com_lojista.total.sum():,.2f}")
print(f"Remessa (50% do spread positivo): R$ {op.comissao_tarvo.sum():,.2f}")
print(f"Total da prévia: R$ {com_lojista.total.sum() + op.comissao_tarvo.sum():,.2f}")

Linhas com contrato encontrado:

 1886 de 1886 (ECs confirmados)
Linhas fora da comissão por EC não confirmado: 57 ( RETIDO )


linha,MDR,antecipacao,total
lojista,,,
Mundo Aberto,17337.07,1185.02,18522.09
Oxford Prep Cursos Ltda,9529.99,6547.91,16077.90
Rota Global Intercâmbio Ltda,14495.62,1125.65,15621.27
Maple Study Assessoria Educacional Ltda,12476.24,2534.54,15010.78
EduAbroad,10312.30,1757.87,12070.17
Escola Bridge de Idiomas Ltda,11916.02,0.00,11916.02
Dublin Hub Intercâmbio Ltda,9390.27,2515.81,11906.08
Canguru,8917.78,2761.65,11679.43
Kiwi Experience,10208.39,503.02,10711.41



Checkout (taxa contratada, ECs confirmados): R$ 207,039.73
Remessa (50% do spread positivo): R$ 39,543.45
Total da prévia: R$ 246,583.18


## 2. Hipóteses

Cada hipótese tem: **o que se afirma**, **o teste** e **o resultado**. O veredito é só sobre o que os arquivos permitem concluir; quando a resposta depende de uma regra do financeiro, fica "inconclusiva" e vira pergunta.

### H1 · A Orbe trunca a taxa em centavos (P2)
*Se for verdade, recalcular com arredondamento comum gera ±R$ 0,01 em centenas de linhas, que não são divergência.*

In [10]:
def acertos(modo):
    n = 0
    for b, t, v in zip(orbe.valor_bruto, orbe.taxa_aplicada_pct, orbe.valor_taxa):
        x = Decimal(str(b)) * Decimal(str(t)) / Decimal(100)
        n += x.quantize(Decimal("0.01"), modo) == Decimal(str(v))
    return n

res = pd.Series({"arredonda (meio para cima)": acertos(ROUND_HALF_UP), "banqueiro": acertos(ROUND_HALF_EVEN),
                 "trunca": acertos(ROUND_DOWN)}, name=f"linhas reproduzidas (de {len(orbe)})")
display(res.to_frame())
print("Linhas em que a taxa aplicada é igual à contratada mas o valor difere R$ 0,01 por causa do truncamento (NÃO é divergência):",
      int(((~base.divergente) & (base.dif_trunc_orbe.abs() > 0)).sum()))
print("Veredito:", "CONFIRMADA" if res["trunca"] == len(orbe) else "NÃO CONFIRMADA")

,linhas reproduzidas (de 1943)
arredonda (meio para cima),942
banqueiro,942
trunca,1943


Linhas em que a taxa aplicada é igual à contratada mas o valor difere R$ 0,01 por causa do truncamento (NÃO é divergência): 855
Veredito: CONFIRMADA


### H2 · A Rota Global foi cobrada +0,50 p.p. a partir de 12/09
*Se for verdade, a divergência começa numa data e atinge todas as faixas de crédito, mas não débito nem Pix.*

In [11]:
rg = base[base.ec == "1074504011"]
div_rg = rg[rg.divergente]
primeira = div_rg.data_taxa_ref.min()
print("Primeira linha divergente:", primeira.date(), "| última:", div_rg.data_taxa_ref.max().date())
print("Antes dessa data:", rg[rg.data_taxa_ref < primeira].divergente.value_counts().to_dict(),
      "| desde essa data:", rg[rg.data_taxa_ref >= primeira].divergente.value_counts().to_dict())
display(pd.crosstab(rg[rg.data_taxa_ref >= primeira].faixa, rg[rg.data_taxa_ref >= primeira].divergente))
dif_pp = (div_rg.taxa_aplicada_pct - div_rg.taxa_contrato_pct).round(2)
print("Diferença em p.p. (aplicada − contratada):", dif_pp.value_counts().to_dict())
print(f"Valor (bruto × Δ taxa): R$ {div_rg.dif_brl.sum():,.2f} em {len(div_rg)} linhas "
      f"(vendas {div_rg[div_rg.tipo_lancamento=='VENDA'].dif_brl.sum():,.2f}; "
      f"estornos {div_rg[div_rg.tipo_lancamento=='ESTORNO'].dif_brl.sum():,.2f})")
print("Veredito: CONFIRMADA. O financeiro confirmou que o contrato da Rota não mudou: é divergência. "
      "Quem arca (devolução ao lojista e cobrança da Orbe) é pergunta em aberto.")

Primeira linha divergente: 2026-09-12 | última: 2026-09-30
Antes dessa data: {False: 53} | desde essa data: {True: 87, False: 19}


divergente,False,True
faixa,,
antecipacao,2,0
credito_2_6x,0,31
credito_7_12x,0,18
credito_a_vista,0,38
debito,7,0
pix,10,0


Diferença em p.p. (aplicada − contratada): {0.5: 87}
Valor (bruto × Δ taxa): R$ 1,546.28 em 87 linhas (vendas 1,644.72; estornos -98.44)
Veredito: CONFIRMADA. O financeiro confirmou que o contrato da Rota não mudou: é divergência. Quem arca (devolução ao lojista e cobrança da Orbe) é pergunta em aberto.


### H3 · A mudança de taxa da Bridge em 15/09 foi aplicada pela Orbe (P3)
*Se a Orbe respeitou a vigência, depois de 15/09 todo crédito à vista da Bridge deveria vir a 2,19%.*

In [12]:
br = base[(base.ec == "1075624862") & (base.faixa == "credito_a_vista")].copy()
br["depois_de_15_09"] = br.data_taxa_ref >= "2026-09-15"
display(br.groupby(["depois_de_15_09", "taxa_aplicada_pct"]).size().rename("linhas").to_frame())
fora = br[br.divergente]
print(f"Linhas fora da vigência: {len(fora)} | diferença (bruto × Δ): R$ {fora.dif_brl.sum():,.2f}")
display(fora[["id_transacao", "data_movimento", "tipo_lancamento", "valor_bruto", "taxa_aplicada_pct", "taxa_contrato_pct",
              "dif_brl"]])
print("Veredito: PARCIAL. A Orbe aplicou 2,19% em parte das linhas e manteve 2,59% em outras depois de 15/09.")

todas = base[base.divergente]
print(f"\nTotal das divergências de checkout (ECs confirmados): {len(todas)} linhas, R$ {todas.dif_brl.sum():,.2f}")
display(todas.groupby("lojista").agg(linhas=("id_transacao", "size"), dif_brl=("dif_brl", "sum")).round(2))

linhas
depois_de_15_09 taxa_aplicada_pct        
False           2.59                    8
True            2.19                    9
                2.59                    5

Linhas fora da vigência: 5 | diferença (bruto × Δ): R$ 29.95


,id_transacao,data_movimento,tipo_lancamento,valor_bruto,taxa_aplicada_pct,taxa_contrato_pct,dif_brl
1099,TX88428053,2026-09-15,VENDA,1124.71,2.59,2.19,4.50
1170,TX88429199,2026-09-16,VENDA,715.68,2.59,2.19,2.86
1216,TX88428930,2026-09-16,VENDA,859.57,2.59,2.19,3.44
1261,TX88429931,2026-09-17,VENDA,1987.36,2.59,2.19,7.95
1327,TX88431123,2026-09-18,VENDA,2801.20,2.59,2.19,11.20


Veredito: PARCIAL. A Orbe aplicou 2,19% em parte das linhas e manteve 2,59% em outras depois de 15/09.

Total das divergências de checkout (ECs confirmados): 210 linhas, R$ 16,973.65


,linhas,dif_brl
lojista,,
Casa do Estudante Viagens Ltda,118,15397.42
Escola Bridge de Idiomas Ltda,5,29.95
Rota Global Intercâmbio Ltda,87,1546.28


### H4 · EC sem mapeamento confirmado deixa muito volume de fora (P4)
*Antes das respostas do financeiro, 4 ECs (290 linhas) estavam sem mapeamento. O que sobra agora?*

In [13]:
peso = (orbe.groupby("status_ec")
        .agg(linhas=("id_transacao", "size"), bruto=("valor_bruto", "sum"), taxa_orbe=("valor_taxa", "sum")).round(2))
peso["pct_do_bruto"] = (peso.bruto / peso.bruto.sum() * 100).round(1)
display(peso)
display(ec[ec.status != "CONFIRMADO"][["ec", "status", "nomes_orbe", "n_linhas_orbe", "aprovado_por", "observacao"]])
confirmados_pelo_financeiro = ec[(ec.fonte_mapeamento == "config/mapeamento_ec.csv") & (ec.status == "CONFIRMADO")]
print(f"ECs confirmados pelo financeiro em config/mapeamento_ec.csv: {len(confirmados_pelo_financeiro)} "
      f"({int(confirmados_pelo_financeiro.n_linhas_orbe.sum())} linhas)")
print("Veredito: RESOLVIDA pelo financeiro para Kiwi, Horizonte Viagens e Novo Horizonte. "
      "Resta a Atlas, retida até o comercial confirmar a taxa (vale desde 10/09, e setembro é faturado com outubro).")

,linhas,bruto,taxa_orbe,pct_do_bruto
status_ec,,,,
CONFIRMADO,1886,7234089.95,224004.31,96.5
RETIDO,57,262138.61,7876.05,3.5


,ec,status,nomes_orbe,n_linhas_orbe,aprovado_por,observacao
18,1076801732,RETIDO,ATLAS IDIOMAS,57,Helena (financeiro),Atlas Idiomas: transacionando desde 10/09 sem contrato assinado na planilha vigente. N...


ECs confirmados pelo financeiro em config/mapeamento_ec.csv: 3 (233 linhas)
Veredito: RESOLVIDA pelo financeiro para Kiwi, Horizonte Viagens e Novo Horizonte. Resta a Atlas, retida até o comercial confirmar a taxa (vale desde 10/09, e setembro é faturado com outubro).


### H4b · Casa do Estudante: tarifa zerada de setembro a dezembro
O financeiro informou que o comercial zerou MDR e antecipação da Casa no checkout, mas a Orbe não foi avisada e cobrou. *O cobrado é devolvido ao lojista.* Com a condição lida de `config/condicoes_comerciais.csv`, a "divergência" da Casa passa a ser a taxa inteira.

In [14]:
casa = base[base.ec == "1018325654"].copy()
print("Taxa contratada aplicada às linhas da Casa em setembro:", sorted(float(x) for x in casa.taxa_contrato_pct.unique()))
casa["origem"] = casa.data_taxa_ref.map(lambda d: "venda de setembro (tarifa zerada)" if d >= pd.Timestamp("2026-09-01") else "venda de agosto (taxa de antes)")
resumo_casa = (casa.groupby(["origem", "linha", "tipo_lancamento"])
               .agg(linhas=("id_transacao", "size"), taxa_cobrada_orbe=("valor_taxa", "sum"),
                    comissao_contrato=("comissao_contrato", "sum"), dif_brl=("dif_brl", "sum")).round(2))
display(resumo_casa)
dev_casa = casa[casa.data_taxa_ref >= pd.Timestamp("2026-09-01")]
print(f"A devolver à Casa (taxa cobrada em vendas de setembro, líquida de estornos desse mês): R$ {dev_casa.valor_taxa.sum():,.2f}")
print(f"Comissão da Casa em setembro: R$ {casa.comissao_contrato.sum():,.2f} "
      "(negativa se só há estorno de venda de agosto). Zero ou negativo não emite nota e vira pendência para o financeiro.")

Taxa contratada aplicada às linhas da Casa em setembro: [0.0, 1.49]


linhas  taxa_cobrada_orbe  comissao_contrato   dif_brl
origem                            linha       tipo_lancamento                                                        
venda de agosto (taxa de antes)   MDR         ESTORNO               1              -5.50               -5.5      0.00
venda de setembro (tarifa zerada) MDR         ESTORNO               1              -9.08                0.0     -9.09
                                              VENDA               113           12578.29                0.0  12578.94
                                  antecipacao ANTECIPACAO           4            2827.56                0.0   2827.57

A devolver à Casa (taxa cobrada em vendas de setembro, líquida de estornos desse mês): R$ 15,396.77
Comissão da Casa em setembro: R$ -5.50 (negativa se só há estorno de venda de agosto). Zero ou negativo não emite nota e vira pendência para o financeiro.


### H5 · O spread realizado pela Ponte difere do contratado para o Mundo Aberto (P5)

In [15]:
def meio_cent(x):
    sinal = -1 if x < 0 else 1
    return sinal * abs(x).quantize(Decimal("0.01"), ROUND_HALF_UP)

# Regra do financeiro: divergência de remessa = valor cheio = (spread % aplicado − contratado) × PTAX × valor em moeda estrangeira.
# O contratado em branco ou "padrão" vale 1,5% (parametros.csv). Tolerância de 0,01 p.p. (parametros.csv) para o arredondamento da Ponte.
op["delta_pp"] = (op.spread_pct - op.spread_contratado_pct).round(4)
op["diverge"] = op.flag_spread_diverge == "S"
op["dif_remessa_brl"] = [float(meio_cent(Decimal(str(d)) / 100 * Decimal(str(p)) * Decimal(str(v)))) if dv else 0.0
                         for d, p, v, dv in zip(op.delta_pp, op.ptax, op.valor_me, op.diverge)]
por_loj = (op.groupby("cnpj")
           .agg(cliente=("cliente_nome", "first"), ops=("num_op", "size"), mediana_realizada_pct=("spread_pct", "median"),
                contratado_pct=("spread_contratado_pct", "first"), ops_divergentes=("diverge", "sum"),
                spread_brl=("spread_brl", "sum")).round(3))
por_loj["regra"] = por_loj.index.map(contrato.drop_duplicates("cnpj").set_index("cnpj").spread_remessa_regra)
display(por_loj.sort_values("ops_divergentes", ascending=False))

d = op[op.diverge].copy()
d["situacao"] = d.spread_brl.map(lambda s: "spread negativo (fora da comissão e da nota)" if s < 0 else "acima do contratado")
display(d.groupby(["cliente_nome", "situacao"]).agg(ops=("num_op", "size"), dif_remessa_brl=("dif_remessa_brl", "sum")).round(2))
ma = d[d.cnpj == "17217888000111"]
print(f"Mundo Aberto: {len(ma)} operações, divergência de valor cheio R$ {ma.dif_remessa_brl.sum():,.2f}. "
      f"A nossa metade (≈ R$ {ma.spread_brl.sum() * (1 - 1.2 / 1.5) / 2:,.2f}) entra na nota de setembro pelo realizado.")
neg = d[d.spread_brl < 0]
print(f"Spreads negativos: {len(neg)} operações; valor cheio contra o contrato R$ {neg.dif_remessa_brl.abs().sum():,.2f}.")
print("Veredito: CONFIRMADA. O financeiro confirmou a regra: Mundo Aberto é divergência a contestar com a Ponte (valor cheio), "
      "e os spreads negativos ficam fora da comissão e da nota.")

,cliente,ops,mediana_realizada_pct,contratado_pct,ops_divergentes,spread_brl,regra
cnpj,,,,,,,
17217888000111,MUNDO ABERTO INTERCÂ,23,1.50,1.20,23,7095.25,EXPLICITO
53857560000123,CANGURU INTERCÂMBIOS,23,1.50,1.50,1,7018.79,PADRAO
68567965000102,DUBLIN HUB INTERCÂMB,14,1.50,1.50,1,3252.87,EXPLICITO
95739378000118,PONTE AÉREA ESTUDANT,15,1.50,1.50,1,5327.03,PADRAO
66564512000199,NOVO HORIZONTE INTER,11,1.40,1.40,1,4069.86,EXPLICITO
00520815000100,CASA DO ESTUDANTE VI,18,1.50,1.50,0,6151.74,EXPLICITO
34671260000100,MAPLE STUDY ASSESSOR,25,1.50,1.50,0,8589.91,EXPLICITO
16181276000153,EDU ABROAD CONSULTOR,16,1.30,1.30,0,5040.46,EXPLICITO
14760206000124,NÓRDICA EDUCAÇÃO INT,18,1.25,1.25,0,5828.25,EXPLICITO


,,ops,dif_remessa_brl
cliente_nome,situacao,,
CANGURU INTERCÂMBIOS,spread negativo (fora da comissão e da nota),1,-1001.30
DUBLIN HUB INTERCÂMB,spread negativo (fora da comissão e da nota),1,-574.90
MUNDO ABERTO INTERCÂ,acima do contratado,23,1418.85
NOVO HORIZONTE INTER,spread negativo (fora da comissão e da nota),1,-637.91
PONTE AÉREA ESTUDANT,spread negativo (fora da comissão e da nota),1,-1351.87


Mundo Aberto: 23 operações, divergência de valor cheio R$ 1,418.85. A nossa metade (≈ R$ 709.53) entra na nota de setembro pelo realizado.
Spreads negativos: 4 operações; valor cheio contra o contrato R$ 3,565.98.
Veredito: CONFIRMADA. O financeiro confirmou a regra: Mundo Aberto é divergência a contestar com a Ponte (valor cheio), e os spreads negativos ficam fora da comissão e da nota.


### H6 · "Spread padrão" da planilha = 1,50% (P8)

In [16]:
padrao = por_loj[por_loj.regra == "PADRAO"]
display(padrao[["cliente", "ops", "mediana_realizada_pct", "contratado_pct"]])
medianas = sorted(float(x) for x in padrao.mediana_realizada_pct.unique())
print("Medianas realizadas dos lojistas com spread 'padrão':", medianas)
print("Veredito:", "CONSISTENTE com 1,50%" if medianas == [1.5] else "NÃO consistente com 1,50%",
      "(é só leitura dos dados da Ponte; a planilha não diz o valor).")

,cliente,ops,mediana_realizada_pct,contratado_pct
cnpj,,,,
07752116000126,KIWI EXPERIENCE TURI,15,1.5,1.5
53857560000123,CANGURU INTERCÂMBIOS,23,1.5,1.5
95739378000118,PONTE AÉREA ESTUDANT,15,1.5,1.5


Medianas realizadas dos lojistas com spread 'padrão': [1.5]
Veredito: CONSISTENTE com 1,50% (é só leitura dos dados da Ponte; a planilha não diz o valor).


### H7 · A operação repetida da Ponte é uma cópia (P7) · e o peso das "com ressalva" (P6)

In [17]:
dup = ponte[ponte.num_op.duplicated(keep=False)]
campos = [c for c in ponte.columns if c not in ("linha_origem", "flag_duplicata")]
diferem = [c for c in campos if dup[c].nunique(dropna=False) > 1]
print("Linhas com o mesmo Nº de operação:", len(dup), "| campos que diferem entre as cópias:", diferem)
display(dup[["num_op", "linha_origem", "valor_brl", "spread_brl", "status_compliance", "observacao", "flag_duplicata"]])
print(f"Spread contado em dobro no Resumo da Ponte: R$ {ponte[ponte.flag_duplicata == 'S'].spread_brl.sum():,.2f}")
print("Veredito: a cópia só difere na observação ('reprocessada'). Provavelmente reprocessamento; confirmar com a Ponte.")

ress = op[op.flag_ressalva == "S"]
print(f"\n'Com ressalva': {len(ress)} operações, spread R$ {ress.spread_brl.sum():,.2f} "
      f"({ress.spread_brl.sum() / op.spread_brl.sum() * 100:.1f}% do spread total) | com texto de motivo: "
      f"{int((ress.observacao != '').sum())}")

Linhas com o mesmo Nº de operação: 2 | campos que diferem entre as cópias: ['observacao']


,num_op,linha_origem,valor_brl,spread_brl,status_compliance,observacao,flag_duplicata
70,PC-0926-0162,76,55600.19,821.88,APROVADO,,N
71,PC-0926-0162,77,55600.19,821.88,APROVADO,reprocessada,S


Spread contado em dobro no Resumo da Ponte: R$ 821.88
Veredito: a cópia só difere na observação ('reprocessada'). Provavelmente reprocessamento; confirmar com a Ponte.

'Com ressalva': 16 operações, spread R$ 4,829.37 (6.3% do spread total) | com texto de motivo: 0


## 3. O buraco do caixa: a Tarvo pagou mais do que recebeu

Na Aurora, os **TEDs da Orbe** (R$ 3,56 mi) são menores que os **PIX de repasse** aos lojistas. Três explicações possíveis; testo o que os dados deixam testar.

In [18]:
au = banco[banco.banco == "AURORA"].copy()
ted = au[au.contraparte.str.contains("ORBE")]
rep = au[(au.tipo_lancamento == "PIX ENVIADO") & ~au.contraparte.str.contains("CONTABILIDADE")]
print(f"TEDs da Orbe recebidos: R$ {ted.valor.sum():,.2f} ({len(ted)})  |  repasses PIX a lojistas: R$ {-rep.valor.sum():,.2f} ({len(rep)})")
print(f"Diferença: R$ {ted.valor.sum() + rep.valor.sum():,.2f}")

TEDs da Orbe recebidos: R$ 3,557,246.29 (21)  |  repasses PIX a lojistas: R$ 4,230,267.61 (44)
Diferença: R$ -673,021.32


### H8 · Defasagem: repasse em ciclos fixos, entrada diária

In [19]:
rep = rep.assign(dia=rep.data.dt.day)
ciclos = rep.groupby("dia").valor.agg(["count", "sum"]).assign(sum=lambda d: -d["sum"]).round(2)
display(ciclos)
rep["ciclo"] = rep.dia.map(lambda d: "ciclo 1 (08–10/09)" if d <= 15 else "ciclo 2 (22–24/09)")
por_ciclo = []
for nome, fim in [("ciclo 1 (08–10/09)", "2026-09-10"), ("ciclo 2 (22–24/09)", "2026-09-24")]:
    por_ciclo.append({"ciclo": nome,
                      "repassado_no_ciclo": round(-rep[rep.ciclo == nome].valor.sum(), 2),
                      "TEDs_Orbe_acumulados_até_o_fim_do_ciclo": round(ted[ted.data <= fim].valor.sum(), 2),
                      "repasses_acumulados": round(-rep[rep.data <= fim].valor.sum(), 2)})
por_ciclo = pd.DataFrame(por_ciclo)
por_ciclo["falta_caixa"] = (por_ciclo.repasses_acumulados - por_ciclo.TEDs_Orbe_acumulados_até_o_fim_do_ciclo).round(2)
display(por_ciclo)
print("Cada lojista recebeu exatamente 1 PIX por ciclo:", bool((rep.groupby("contraparte").size() == 2).all()))
print("Veredito: a defasagem EXPLICA o vermelho no meio do mês. O financeiro confirmou o mecanismo: o saldo negativo é a "
      "conta garantida do Aurora, usada para pagar os lojistas (2 vezes por mês) antes de a Orbe liquidar o crédito e o parcelado.")
print("Ponto em aberto: a regra diz repassar 'o que a Orbe já liquidou até ali', mas no ciclo 1 foram repassados "
      f"R$ {-rep[rep.ciclo == 'ciclo 1 (08–10/09)'].valor.sum():,.2f} com R$ {ted[ted.data <= '2026-09-10'].valor.sum():,.2f} recebidos.")

,count,sum
dia,,
8,7,582223.70
9,5,441270.64
10,10,1084324.37
22,4,302078.35
23,9,872462.70
24,9,947907.85


,ciclo,repassado_no_ciclo,TEDs_Orbe_acumulados_até_o_fim_do_ciclo,repasses_acumulados,falta_caixa
0,ciclo 1 (08–10/09),2107818.71,1039172.54,2107818.71,1068646.17
1,ciclo 2 (22–24/09),2122448.90,2832520.79,4230267.61,1397746.82


Cada lojista recebeu exatamente 1 PIX por ciclo: True
Veredito: a defasagem EXPLICA o vermelho no meio do mês. O financeiro confirmou o mecanismo: o saldo negativo é a conta garantida do Aurora, usada para pagar os lojistas (2 vezes por mês) antes de a Orbe liquidar o crédito e o parcelado.
Ponto em aberto: a regra diz repassar 'o que a Orbe já liquidou até ali', mas no ciclo 1 foram repassados R$ 2,107,818.71 com R$ 1,039,172.54 recebidos.


### H9 · A Tarvo financia antecipações com caixa próprio
*Se for verdade, quem antecipou deveria ter recebido PIX acima do líquido das vendas do mês.*

O mapa abaixo liga o nome do beneficiário ao EC **só para este teste** (hipótese explícita, não é usado em nenhuma tabela).

In [20]:
MAPA = {"ATLAS IDIOMAS": "1076801732", "EDU ABROAD": "1020660828", "MAPLE STUDY": "1041180753", "BEM-VINDO": "1098729012",
        "VELA TURISMO": "1086291869", "SOL NASCENTE": "1064304917", "OXFORD": "1000960410", "CANGURU": "1061760196",
        "KIWI": "1038867654", "FLUENTE": "1066100982", "ROTA GLOBAL": "1074504011", "DUBLIN": "1014537966",
        "NOVO HORIZONTE": "1043300981", "MUNDO ABERTO": "1037647039", "TRILHA": "1068443135", "BAGAGEM": "1087276103",
        "CAMPUS": "1070531381", "LISBOA": "1023622151", "CASA DO ESTUDANTE": "1018325654", "ANDES": "1058961395",
        "ESCOLA BRIDGE": "1075624862", "HORIZONTE VIAGENS": "1043396662"}
rep["ec_hipotese"] = rep.contraparte.map(lambda n: next((v for k, v in MAPA.items() if k in n.upper()), None))
print("Repasses sem EC pelo mapa:", int(rep.ec_hipotese.isna().sum()))
orbe["grupo"] = orbe.tipo_lancamento.map(lambda t: "antecipacao" if "ANTECIPACAO" in t else "vendas_estornos")
liq = orbe.pivot_table(index="ec", columns="grupo", values="valor_liquido", aggfunc="sum", fill_value=0)
t = liq.join((-rep.groupby("ec_hipotese").valor.sum()).rename("pix_recebido"), how="outer").fillna(0)
t.insert(0, "lojista", t.index.map(nome_ec))
t["pix_menos_vendas"] = t.pix_recebido - t.vendas_estornos
t["pix_menos_vendas_e_antecipacao"] = t.pix_recebido - t.vendas_estornos - t.antecipacao
display(t.round(0).sort_values("antecipacao", ascending=False))
com = t[t.antecipacao > 0]
n_acima = int((com.pix_menos_vendas >= 0).sum())
print(f"Dos {len(com)} lojistas que antecipam, receberam PIX >= líquido das vendas: {n_acima}")
print(f"Em linha com vendas + antecipação (|diferença| < R$ 10 mil): {int((com.pix_menos_vendas_e_antecipacao.abs() < 10000).sum())}")
print("Veredito:", "NÃO SUSTENTADA" if n_acima <= 1 else "SUSTENTADA",
      "pelos dados: a maioria dos lojistas que antecipou recebeu MENOS que o líquido das vendas do mês. "
      "O Atlas é a exceção (recebeu vendas + antecipação).")

Repasses sem EC pelo mapa: 0


,lojista,antecipacao,vendas_estornos,pix_recebido,pix_menos_vendas,pix_menos_vendas_e_antecipacao
ec,,,,,,
1000960410,Oxford Prep Cursos Ltda,305257.0,290576.0,239547.0,-51029.0,-356286.0
1018325654,Casa do Estudante Viagens Ltda,131819.0,372468.0,113334.0,-259134.0,-390953.0
1061760196,Canguru,128745.0,264631.0,234978.0,-29654.0,-158399.0
1076801732,ATLAS IDIOMAS,124559.0,129703.0,249302.0,119598.0,-4961.0
1041180753,Maple Study Assessoria Educacional Ltda,118158.0,363090.0,330031.0,-33059.0,-151217.0
1014537966,Dublin Hub Intercâmbio Ltda,117285.0,286138.0,176479.0,-109659.0,-226944.0
1020660828,EduAbroad,81950.0,316661.0,62037.0,-254624.0,-336574.0
1037647039,Mundo Aberto,55244.0,535917.0,137461.0,-398456.0,-453701.0
1074504011,Rota Global Intercâmbio Ltda,52477.0,454589.0,207556.0,-247033.0,-299509.0


Dos 10 lojistas que antecipam, receberam PIX >= líquido das vendas: 1
Em linha com vendas + antecipação (|diferença| < R$ 10 mil): 1
Veredito: NÃO SUSTENTADA pelos dados: a maioria dos lojistas que antecipou recebeu MENOS que o líquido das vendas do mês. O Atlas é a exceção (recebeu vendas + antecipação).


### H10 · O "buraco" é o parcelamento: a Orbe liquida em parcelas
*Se verdade, a soma que a Orbe devia pagar em setembro é bem menor que o líquido reportado.* Os dados não trazem a regra de liquidação (D+30? mensal?), então só dá para ver a **ordem de grandeza**.

In [21]:
vend = orbe[orbe.tipo_lancamento.isin(["VENDA", "ESTORNO"])]
parc = vend.parcelas.fillna(1)
print(f"Líquido reportado (vendas + estornos): R$ {vend.valor_liquido.sum():,.2f}")
print(f"Se só a 1ª parcela de cada venda liquidasse no mês: R$ {(vend.valor_liquido / parc).sum():,.2f}")
print(f"TEDs de fato recebidos da Orbe: R$ {ted.valor.sum():,.2f}")
print(f"Parcelado concentra {vend[vend.modalidade == 'CREDITO PARCELADO'].valor_liquido.sum() / vend.valor_liquido.sum() * 100:.0f}% do líquido reportado.")
import itertools

# Há uma regra exata? Testa se algum TED é a soma de componentes do dia (pix, débito, à vista, parcelado, antecipação, estorno),
# com atraso de 0 a 4 dias, usando o líquido integral ou só a 1ª parcela.
o2 = orbe.assign(comp=orbe.apply(lambda r: "antecipacao" if "ANTECIPACAO" in r.tipo_lancamento
                                 else "estorno" if r.tipo_lancamento == "ESTORNO" else r.faixa, axis=1),
                 liq_1a=orbe.valor_liquido / orbe.parcelas.fillna(1))
comps = ["pix", "debito", "credito_a_vista", "credito_2_6x", "credito_7_12x", "antecipacao", "estorno"]
tabs = [o2.pivot_table(index="data_movimento", columns="comp", values=v, aggfunc="sum", fill_value=0).reindex(columns=comps, fill_value=0)
        for v in ("valor_liquido", "liq_1a")]
explicados = 0
for _, r in ted.iterrows():
    for lag in range(5):
        dref = r["data"] - pd.Timedelta(days=lag)
        if any(dref in t.index and abs(t.loc[dref, list(sub)].sum() - r.valor) < 0.02
               for t in tabs for k in range(1, 5) for sub in itertools.combinations(comps, k)):
            explicados += 1
            break
razao = (ted.set_index("data").valor / vend.groupby("data_movimento").valor_liquido.sum().reindex(ted.data).values)
print(f"TEDs que são a soma exata de componentes do dia (atraso 0 a 4 dias): {explicados} de {len(ted)}")
print(f"Razão TED / líquido do dia: de {razao.min():.2f} a {razao.max():.2f} (não é constante)")

piso, teto = (vend.valor_liquido / parc).sum(), vend.valor_liquido.sum()
print(f"Veredito: {'COMPATÍVEL' if piso < ted.valor.sum() < teto else 'INCOMPATÍVEL'}. "
      f"Os TEDs ficam entre o piso (R$ {piso:,.0f}) e o teto (R$ {teto:,.0f}), mas incluem também parcelas de vendas "
      "de meses anteriores, que não temos. O financeiro confirmou: débito e Pix caem no dia seguinte, crédito e parcelado "
      "ao longo dos meses; a conciliação parcela a parcela fica para quando houver a agenda de recebíveis.")

Líquido reportado (vendas + estornos): R$ 6,125,403.55
Se só a 1ª parcela de cada venda liquidasse no mês: R$ 2,161,542.94
TEDs de fato recebidos da Orbe: R$ 3,557,246.29
Parcelado concentra 81% do líquido reportado.


TEDs que são a soma exata de componentes do dia (atraso 0 a 4 dias): 0 de 21
Razão TED / líquido do dia: de 0.29 a 1.80 (não é constante)
Veredito: COMPATÍVEL. Os TEDs ficam entre o piso (R$ 2,161,543) e o teto (R$ 6,125,404), mas incluem também parcelas de vendas de meses anteriores, que não temos. O financeiro confirmou: débito e Pix caem no dia seguinte, crédito e parcelado ao longo dos meses; a conciliação parcela a parcela fica para quando houver a agenda de recebíveis.


### H11 · Remessa: o que entra dos lojistas é o que sai para a Ponte

In [22]:
import re
import unicodedata

me = banco[banco.banco == "MERIDIANO"].copy()
entra = me[me.tipo_lancamento == "PIX RECEB"].valor.sum()
ted_p = me[me.tipo_lancamento == "TED ENVIADA"].copy()
ted_p["valor_ted"] = (-ted_p.valor).round(2)
op["total_pago_ponte"] = (op.valor_brl + op.iof_brl).round(2)
print(f"PIX recebido dos lojistas: R$ {entra:,.2f}  |  TEDs para a Ponte: R$ {ted_p.valor_ted.sum():,.2f}  |  "
      f"diferença: R$ {entra - ted_p.valor_ted.sum():,.2f}")
print(f"Soma de (Valor R$ + IOF) das operações únicas da Ponte: R$ {op.total_pago_ponte.sum():,.2f}")

por_dia = op.groupby("data_contratacao").agg(n_ops=("total_pago_ponte", "size"), soma_ops=("total_pago_ponte", "sum")).round(2)
conf_ted = ted_p.merge(por_dia, left_on="data", right_index=True, how="left")
conf_ted["bate"] = (conf_ted.valor_ted - conf_ted.soma_ops).abs() < 0.01
display(conf_ted[["data", "ref_origem", "valor_ted", "n_ops", "soma_ops", "bate"]].reset_index(drop=True))
print("TEDs que fecham, ao centavo, com as operações contratadas no mesmo dia:", int(conf_ted.bate.sum()), "de", len(conf_ted))
dup_pago = ponte[ponte.flag_duplicata == "S"]
print(f"Se a operação repetida também tivesse sido paga, o total seria R$ {op.total_pago_ponte.sum() + (dup_pago.valor_brl + dup_pago.iof_brl).sum():,.2f}; "
      "o banco mostra que foi paga uma vez só.")
print("As operações com spread negativo estão dentro desses TEDs:",
      bool(op[op.spread_brl < 0].total_pago_ponte.sum() > 0), "(a Tarvo paga a Ponte pela operação inteira, o spread é outra conta)")
print("Veredito:", "CONFIRMADA" if int(conf_ted.bate.sum()) == len(conf_ted) else "PARCIAL",
      "- cada TED à Ponte é a soma de (Valor R$ + IOF) das operações do dia.")

PIX recebido dos lojistas: R$ 5,629,687.12  |  TEDs para a Ponte: R$ 5,611,447.12  |  diferença: R$ 18,240.00
Soma de (Valor R$ + IOF) das operações únicas da Ponte: R$ 5,611,447.12


,data,ref_origem,valor_ted,n_ops,soma_ops,bate
0,2026-09-01,MRD2026090094,240619.49,11,240619.49,True
1,2026-09-02,MRD2026090116,383630.51,10,383630.51,True
2,2026-09-03,MRD2026090120,259289.88,10,259289.88,True
3,2026-09-04,MRD2026090138,491127.36,17,491127.36,True
4,2026-09-08,MRD2026090246,264853.10,13,264853.10,True
5,2026-09-09,MRD2026090264,330847.90,13,330847.90,True
6,2026-09-10,MRD2026090272,258911.16,9,258911.16,True
7,2026-09-11,MRD2026090286,362392.13,13,362392.13,True
8,2026-09-14,MRD2026090295,139283.36,8,139283.36,True
9,2026-09-15,MRD2026090400,467019.61,18,467019.61,True


TEDs que fecham, ao centavo, com as operações contratadas no mesmo dia: 21 de 21
Se a operação repetida também tivesse sido paga, o total seria R$ 5,667,258.59; o banco mostra que foi paga uma vez só.
As operações com spread negativo estão dentro desses TEDs: True (a Tarvo paga a Ponte pela operação inteira, o spread é outra conta)
Veredito: CONFIRMADA - cada TED à Ponte é a soma de (Valor R$ + IOF) das operações do dia.


**E o excesso de PIX recebido?** Os lojistas mandaram mais do que a Ponte cobrou. Por lojista (ligando o pagador ao cliente da Ponte pela primeira palavra do nome, só neste teste):

In [23]:
def primeiro_token(s):
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode().upper()
    return re.split(r"[^A-Z0-9]+", s.strip())[0]


tok_cnpj = {primeiro_token(n): c for c, n in op.groupby("cnpj").cliente_nome.first().items()}
assert len(tok_cnpj) == op.cnpj.nunique(), "primeiras palavras repetidas entre clientes da Ponte"
pix_loj = me[me.tipo_lancamento == "PIX RECEB"].copy()
pix_loj["cnpj"] = pix_loj.contraparte.map(lambda n: tok_cnpj.get(primeiro_token(n)))
print("PIX de pagadores que não são clientes da Ponte:", int(pix_loj.cnpj.isna().sum()))
dev = op.groupby("cnpj").agg(cliente=("cliente_nome", "first"), operacoes=("num_op", "size"), ops_total=("total_pago_ponte", "sum"))
dev["pix_recebido"] = pix_loj.groupby("cnpj").valor.sum()
dev["pix_menos_ops"] = (dev.pix_recebido - dev.ops_total).round(2)
display(dev.round(2).sort_values("pix_menos_ops"))
print(f"Soma das diferenças: R$ {dev.pix_menos_ops.sum():,.2f}")

PIX de pagadores que não são clientes da Ponte: 0


,cliente,operacoes,ops_total,pix_recebido,pix_menos_ops
cnpj,,,,,
00520815000100,CASA DO ESTUDANTE VI,18,417843.39,417843.39,0.0
07752116000126,KIWI EXPERIENCE TURI,15,362552.48,362552.48,0.0
14760206000124,NÓRDICA EDUCAÇÃO INT,18,473909.27,473909.27,0.0
16181276000153,EDU ABROAD CONSULTOR,16,394222.69,394222.69,0.0
17217888000111,MUNDO ABERTO INTERCÂ,23,481950.02,481950.02,0.0
34671260000100,MAPLE STUDY ASSESSOR,25,583530.13,583530.13,0.0
50900987000179,BAGAGEM LIVRE TURISM,8,175575.09,175575.09,0.0
53857560000123,CANGURU INTERCÂMBIOS,23,543693.33,543693.33,0.0
61720397000180,LISBOA CURSOS E VIAG,16,407463.55,407463.55,0.0


Soma das diferenças: R$ 18,240.00


### H12 · Toda transferência Aurora → Meridiano aparece nos dois extratos

In [24]:
a_tr = banco[(banco.banco == "AURORA") & banco.contraparte.str.startswith("TARVO")][["ref_origem", "data", "valor"]].copy()
a_tr["chave"] = a_tr.valor.abs().round(2)
m_tr = banco[(banco.banco == "MERIDIANO") & banco.contraparte.str.startswith("TARVO")][["ref_origem", "data", "valor"]].copy()
m_tr["chave"] = m_tr.valor.abs().round(2)
cruz = a_tr.merge(m_tr, on=["data", "chave"], how="outer", suffixes=("_aurora", "_meridiano"), indicator=True)
display(cruz[["data", "ref_origem_aurora", "valor_aurora", "ref_origem_meridiano", "valor_meridiano", "_merge"]])
print("Sem contrapartida:", int((cruz._merge != "both").sum()))
print("Veredito: REFUTADA. A saída de R$ 300.000 de 30/09 não tem entrada no Meridiano em setembro (em trânsito?).")

,data,ref_origem_aurora,valor_aurora,ref_origem_meridiano,valor_meridiano,_merge
0,2026-09-02,L12,-350000.0,MRD2026090107,350000.0,both
1,2026-09-11,L44,-420000.0,MRD2026090274,420000.0,both
2,2026-09-21,L58,-380000.0,MRD2026090454,380000.0,both
3,2026-09-30,L92,-300000.0,NaN,NaN,left_only


Sem contrapartida: 1
Veredito: REFUTADA. A saída de R$ 300.000 de 30/09 não tem entrada no Meridiano em setembro (em trânsito?).


## 4. O que fica

Resumo calculado a partir das saídas acima (setembro/2026), já com as respostas do financeiro. O que ainda depende dele está em `PERGUNTAS_FINANCEIRO.md`.

In [25]:
from IPython.display import Markdown, display

sem_nota = com_lojista[com_lojista.total <= 0]
com_nota = com_lojista[com_lojista.total > 0]
atlas = orbe[orbe.status_ec == "RETIDO"]
div_loj = todas.groupby("lojista").dif_brl.sum().round(2).sort_values(ascending=False)
lista_div = "; ".join(f"{k}: R$ {v:,.2f}" for k, v in div_loj.items())
mundo = d[d.cnpj == "17217888000111"]
negativos = d[d.spread_brl < 0]
txt = f"""
### A conta do mês (prévia)
- **Checkout**: R$ {com_nota.total.sum():,.2f} de comissão pela taxa contratada, em {len(com_nota)} lojistas com nota a emitir (MDR + antecipação, na mesma nota).
- **Sem nota (zero ou negativo → pendência para o financeiro)**: {', '.join(f"{k} (R$ {v:,.2f})" for k, v in sem_nota.total.items()) or 'nenhum'}.
- **Fora do cálculo**: {len(atlas)} linhas da Atlas, retida até o comercial confirmar a taxa (vale desde 10/09, e setembro é faturado junto com outubro).
- **Remessa**: R$ {op.comissao_tarvo.sum():,.2f} em uma nota única para a Ponte Câmbio (50% do spread positivo de {len(op)} operações; {int((op.spread_brl < 0).sum())} spreads negativos ficam fora).
- **Total da prévia**: R$ {com_nota.total.sum() + op.comissao_tarvo.sum():,.2f}.
- **Divergências de checkout** (bruto × Δ taxa): R$ {todas.dif_brl.sum():,.2f} em {len(todas)} linhas. {lista_div}.
- **Divergências de remessa** (valor cheio, a contestar com a Ponte): Mundo Aberto R$ {mundo.dif_remessa_brl.sum():,.2f} em {len(mundo)} operações; spreads negativos R$ {negativos.dif_remessa_brl.abs().sum():,.2f} em {len(negativos)} operações.
"""
display(Markdown(txt))


### A conta do mês (prévia)
- **Checkout**: R$ 207,045.23 de comissão pela taxa contratada, em 20 lojistas com nota a emitir (MDR + antecipação, na mesma nota).
- **Sem nota (zero ou negativo → pendência para o financeiro)**: Casa do Estudante Viagens Ltda (R$ -5.50).
- **Fora do cálculo**: 57 linhas da Atlas, retida até o comercial confirmar a taxa (vale desde 10/09, e setembro é faturado junto com outubro).
- **Remessa**: R$ 39,543.45 em uma nota única para a Ponte Câmbio (50% do spread positivo de 232 operações; 4 spreads negativos ficam fora).
- **Total da prévia**: R$ 246,588.68.
- **Divergências de checkout** (bruto × Δ taxa): R$ 16,973.65 em 210 linhas. Casa do Estudante Viagens Ltda: R$ 15,397.42; Rota Global Intercâmbio Ltda: R$ 1,546.28; Escola Bridge de Idiomas Ltda: R$ 29.95.
- **Divergências de remessa** (valor cheio, a contestar com a Ponte): Mundo Aberto R$ 1,418.85 em 23 operações; spreads negativos R$ 3,565.98 em 4 operações.


### Hipóteses
| # | Hipótese | Veredito |
|---|---|---|
| H1 | A Orbe trunca a taxa em centavos | **Confirmada** nos dados e pelo financeiro ("a Orbe trunca, a gente não"). Diferença de centavo não é divergência |
| H2 | Rota Global cobrada +0,50 p.p. desde 12/09 | **Confirmada**. O financeiro disse que o contrato não mudou: é divergência. Quem arca é **pergunta** |
| H3 | Bridge aplicada na vigência de 15/09 | **Parcial**: 5 vendas depois de 15/09 ainda a 2,59%. Quem arca é **pergunta** |
| H4 | ECs sem mapeamento deixam volume de fora | **Resolvida** pelo financeiro para Kiwi, Horizonte Viagens e Novo Horizonte. Resta a Atlas (retida) |
| H4b | Casa do Estudante com tarifa zerada | **Confirmada pelo financeiro** (set a dez, MDR e antecipação). O que a Orbe cobrou é devolvido ao lojista |
| H5 | Spread do Mundo Aberto difere do contratado | **Confirmada**: divergência de valor cheio a contestar com a Ponte; a nota de setembro sai pelo realizado |
| H6 | Spread "padrão" = 1,50% | **Confirmada pelo financeiro** |
| H7 | A operação repetida da Ponte é cópia | **Confirmada**: conta uma vez e entra na lista de correção para a Ponte. O banco mostra que foi paga uma vez |
| H8 | O buraco do caixa é defasagem | **Confirmada pelo financeiro**: é a conta garantida do Aurora, usada para repassar antes de a Orbe liquidar. Fica em aberto a regra do valor repassado |
| H9 | A Tarvo financia antecipações com caixa próprio | **Não sustentada** pelos dados (9 de 10 antecipadores receberam menos que o líquido das vendas) |
| H10 | O buraco é o parcelamento | **Compatível**: débito e Pix caem em D+1, crédito e parcelado ao longo dos meses (financeiro). Conciliar parcela a parcela fica para a agenda de recebíveis |
| H11 | A remessa fecha entre extrato e relatório | **Confirmada, ao centavo** |
| H12 | Toda transferência entre contas aparece nos dois bancos | **Refutada**, mas explicada: os R$ 300.000 de 30/09 caem no Meridiano em 01/10 (financeiro) |

### Dois lados do caixa
- **Remessa fecha por completo.** Cada um dos 21 TEDs para a Ponte é a soma de (Valor R$ + IOF) das operações do mesmo dia, e o PIX de cada lojista é a soma das suas operações (13 de 14 exatos). O TED aponta, com rastro, para os `Nº Operação` do dia.
- **Checkout não fecha com o banco**, e o financeiro disse que não é preciso fechar neste mês: os créditos da Orbe são classificados, e o resto espera a agenda de recebíveis.

### Valores redondos: o que o financeiro respondeu
| Valor | Onde | Resposta |
|---|---|---|
| +R$ 18.240,00 | Meridiano, 17/09, PIX da Ponte Aérea | É dinheiro de uma remessa que sai em outubro (recursos de lojistas a liquidar) |
| +R$ 7.350,00 | Aurora, 16/09, PIX sem identificação | Não é receita até alguém identificar |
| +R$ 12.400,00 | Aurora, 23/09, devolução de PIX do Mundo Aberto | É um PIX de repasse que voltou; o motivo ainda não se sabe |
| −R$ 300.000,00 | Aurora, 30/09, transferência ao Meridiano | Sai do Aurora em 30/09 e cai no Meridiano em 01/10 |

## Nota de revisão (17/10, Gabriel; ver `HIPOTESES.md`)

Estas análises foram feitas antes de algumas respostas da Helena. Para a saída final valem estas correções, **sem alterar as células acima**:

- **Casa do Estudante fora das divergências.** A Helena disse que a devolução da Casa fica *separada* das divergências de contrato (R28). As divergências de contrato de checkout são **R$ 1.576,23 em 92 linhas** (Rota Global R$ 1.546,28; Bridge R$ 29,95). A Casa (118 linhas) é devolução ao lojista, no valor **cobrado** pela Orbe: **R$ 15.396,77**, e não os R$ 15.397,42 de bruto × taxa.
- **Mundo Aberto: R$ 1.418,88** (valor realizado, `Spread R$` − contratado × Valor ME × PTAX, em `Decimal`), e não R$ 1.418,85 (P13). A diferença é o arredondamento do `spread_pct` do relatório.
- **Tolerância do spread: 0,005 p.p.** (P1), não 0,01. Resultado idêntico em setembro (27 operações).
- **Ajuste da Ponte (R27)** passou a ser regra confirmada; não é mais pendência (P9 superada).